In [1]:
# Baseline Capacitance Simulation (Transmon Only)
from pathlib import Path
from qiskit_metal import designs, MetalGUI
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.toolbox_metal.parsing import parse_units
from qiskit_metal.renderers.renderer_qtcad.qtcad_renderer import QQTCADRenderer
import numpy as np

In [2]:
# --- PARAMETER SWEEP SETTINGS ---
width_adjustment = 0
height_adjustment = 0

pad_width_total = f"{500 + width_adjustment}um"
pad_height_total = f"{215 + height_adjustment}um"

print(f"Sweep Params: Width={pad_width_total}, Height={pad_height_total}")

# Set up directory
device_name = "transmon_baseline_cap"
output_dir = str(Path.cwd() / "output" / device_name)
geo_filepath = output_dir + "/" + device_name + ".xao"
mesh_filepath = output_dir + "/" + device_name + ".msh4"

Sweep Params: Width=500um, Height=215um


In [3]:
tolerance_relative = 0.05
mesh_h_min = "15um"
mesh_h_max = "150um"

In [4]:
design = designs.MultiPlanar({}, True)
design.overwrite_enabled = True

transmon_label = "Q1"

# Set up transmon pocket with connection pad for baseline
options_qubit = dict(
    pos_x='0um',
    pos_y='0um',
    pad_gap='30um',
    pad_width=pad_width_total,
    pad_height=pad_height_total,
    pocket_width='700um',
    pocket_height='700um',
    orientation="0",
    connection_pads=dict(
        coupler=dict(loc_W=1, loc_H=-1, pad_width='70um', cpw_extend='50um')
    )
)

qubit = TransmonPocket(design, transmon_label, options=options_qubit)

# Mark the coupler pin as open for the simulation
open_pins = [(transmon_label, 'coupler')]

In [5]:
gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

In [ ]:
gui.main_window.close()

In [6]:
# --- QTCAD CAPACITANCE EXTRACTION ---
qtcad_renderer = QQTCADRenderer(
    design,
    options=dict(
        adaptive=True,
        output_dir=output_dir,
        geo_filepath=geo_filepath,
        mesh_filepath=mesh_filepath,
        mesh_scale=1e-3,
        adaptive_mesh_scale=1e-3,
        make_subdir=False,
        capacitance=dict(tol_rel=0.05, tol_abs=0.0),
    ),
)

qtcad_renderer.render_design(
    open_pins=[],
    skip_junctions=True,
    mesh_geoms=False,
    initial_mesh_h_min=mesh_h_min,
    initial_mesh_h_max=mesh_h_max,
)

qtcad_renderer.gmsh.add_mesh(dim=3, intelli_mesh=False)
qtcad_renderer.export_mesh()
qtcad_renderer.export_parameters()

print("\n--- Running QTCAD Capacitance Extraction ---")
qtcad_renderer.run_qtcad("cap")
cap_matrix = qtcad_renderer.load_qtcad_capacitance_matrix()
print("Capacitance Matrix (fF):")
print(cap_matrix)


--- Running QTCAD Capacitance Extraction ---


10:30AM 05s INFO [run_qtcad]: =================
10:30AM 05s INFO [run_qtcad]: Running QTCAD®...
10:30AM 05s INFO [run_qtcad]: =================



|  ||  |||                                                                 \\  
|  ||  |||         @@@@     @@@@@@@@      @@@@@        @       @@@@@@@      \\ 
|  ||  |||      @@@   @@@      @@      @@@    @       @@       @@    @@@     \\
|  ||  |||     @@       @@     @@     @@             @ @@      @@     @@@     \\
|  ||  |||     @@       @@     @@     @@            @    @     @@      @@     //
|  ||  |||      @@     @@      @@      @@@    @    @@@@@@@@    @@    @@@     //
|  ||  |||        @@@@@@       @@        @@@@@@   @@      @@   @@@@@@       // 
|  ||  |||             @@                                                  //   

                                 Version 2.1.3                                  
  Copyright (c) 2022-2026 Nanoacademic Technologies Inc. All rights reserved.   

      Welcome to QTCAD, the Quantum-Technology Computer-Aided Design tool.      

                        For documentation, please visit:                        
                      https:/